<a href="https://colab.research.google.com/github/PRR-aiexp/CVYoloMLops/blob/main/CVYoloMLops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/PRR-aiexp/CVYoloMlops.git
#cd CVYoloMlops

fatal: destination path 'CVYoloMlops' already exists and is not an empty directory.


In [3]:
!pip install ultralytics opencv-python notebook transformers mlflow

In [4]:
!pip install dagshub mlflow ultralytics pandas pyyaml

In [5]:
import mlflow
import dagshub
from dagshub import init
from dagshub.auth import add_app_token

In [6]:
DAGSHUB_TOKEN = "2d85e0296e3911b8dd7c7310f5b0968a3b4541b8"  # paste your token

add_app_token(DAGSHUB_TOKEN)

In [7]:
REPO_OWNER = "PRR-aiexp"   # e.g. "PRR-aiexp"
REPO_NAME  = "CVYoloMlops"       # e.g. "CVYoloMlops"

init(
    repo_owner=REPO_OWNER,
    repo_name=REPO_NAME,
    mlflow=True,   # <-- this wires MLflow for you
    dvc=False
)

print("Tracking URI now:", mlflow.get_tracking_uri())

Accessing as PRR-aiexp

Initialized MLflow to track repo "PRR-aiexp/CVYoloMlops"

Repository PRR-aiexp/CVYoloMlops initialized!

Tracking URI now: https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow


In [8]:
from ultralytics import YOLO
import mlflow
import yaml
import os

# End any active MLflow runs before starting a new one
if mlflow.active_run():
    mlflow.end_run()

# Update the path to the dataset YAML file to reflect the cloned repository structure
DATASET_YAML = "CVYoloMlops/data/yolo_dataset/car_detection.yaml"

# Define the absolute paths for train and validation images and labels
repo_root = "/content/CVYoloMlops/"
yolo_dataset_root = os.path.join(repo_root, "data", "yolo_dataset")

train_images_path = os.path.join(yolo_dataset_root, "images", "train")
val_images_path = os.path.join(yolo_dataset_root, "images", "val")
train_labels_path = os.path.join(yolo_dataset_root, "labels", "train")
val_labels_path = os.path.join(yolo_dataset_root, "labels", "val")

# Create dummy directories to satisfy Ultralytics' initial checks
# In a real scenario, you would have your actual dataset files here.
os.makedirs(train_images_path, exist_ok=True)
os.makedirs(val_images_path, exist_ok=True)
os.makedirs(train_labels_path, exist_ok=True)
os.makedirs(val_labels_path, exist_ok=True)

# New content for car_detection.yaml with absolute paths for robustness
dataset_config = {
    'train': train_images_path,
    'val': val_images_path,
    'nc': 1, # number of classes, assuming 'car' is the only class
    'names': ['car']
}

# Write the updated dataset configuration to the YAML file in the cloned repo
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_config, f)

# Set MLflow experiment name explicitly
mlflow.set_experiment("YOLOv8 Car Detection Experiment")

with mlflow.start_run(run_name="yolov8n_colab_run1"):

    # Log parameters
    mlflow.log_param("model", "yolov8n")
    mlflow.log_param("imgsz", 640)
    mlflow.log_param("epochs", 5)   # start small
    mlflow.log_param("batch", 8)

    model = YOLO("yolov8n.pt")

    # Define project and name for the training run
    train_project = "runs/train"
    train_name = "car_yolo_dagshub_test"

    results = model.train(
        data=DATASET_YAML,
        imgsz=640,
        epochs=5,        # small test first
        batch=8,
        device=0,        # GPU; use "cpu" if no GPU
        project=train_project,
        name=train_name,
        exist_ok=True
    )

    # Log YOLO metrics
    metrics = results.results_dict
    for k, v in metrics.items():
        try:
            mlflow.log_metric(k, float(v))
        except Exception:
            pass

    # Log best model weights as artifact
    # The best model path is constructed from the project and name of the training run
    best_model_path = os.path.join("/content", train_project, train_name, "weights", "best.pt")
    mlflow.log_artifact(best_model_path)

print("Done; run should now be in DagsHub MLflow UI.")

Ultralytics 8.3.235 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=CVYoloMlops/data/yolo_dataset/car_detection.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=car_yolo_dagshub_test, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, persp

2025/12/09 03:50:08 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2025/12/09 03:50:08 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(a32e67dfd59741ed99b5e8b66b8828f0) to https://dagshub.com/PRR-aiexp/CVYoloMlops.mlflow
MLflow: disable with 'yolo settings mlflow=False'
WARNING ⚠️ MLflow: Failed to initialize: INVALID_PARAMETER_VALUE: Response: {'error_code': 'INVALID_PARAMETER_VALUE'}
WARNING ⚠️ MLflow: Not tracking this run
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/train/car_yolo_dagshub_test
Starting training for 5 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        1/5      1.09G      1.454      2.382      1.111          5        640: 100% ━━━━━━━━━━━━ 36/36 4.0it/s 9.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.5it/s 1.4s
                   all         71        110    0.00512      0.991        0.9      0.446

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
        2/5      1.16G      1.329      1.